In [ ]:
import datetime as dt
from edgar import Company, set_identity
import time
from lxml import etree # type: ignore
from httpx import HTTPStatusError

MAX_RETRIES = 3
VALID_FORMS = ["13F-HR", "13F-HR/A"]

set_identity( "example2.company@access.com")

def gather_holdings_from_soup(_, tables, ns):
    """Parse XML soup and return list of Holding objects"""
    holdings = []

    for table in tables:
        print(etree.tostring(table, pretty_print=True))
        # Extract basic data
        name_of_issuer = table.xpath('string(ns:nameOfIssuer)', namespaces=ns).strip()
        cusip = table.xpath('string(ns:cusip)', namespaces=ns).strip()
        title_of_class = table.xpath('string(ns:titleOfClass)', namespaces=ns).strip()

        try:
            value = int(float(table.xpath('string(ns:value)', namespaces=ns).strip()))
            print(value)

            shares_amount = int(float(table.xpath('string(ns:shrsOrPrnAmt/ns:sshPrnamt)', namespaces=ns).strip() or 0))
            print("share amount: ", shares_amount)
        except (ValueError, AttributeError) as e:
            raise ValueError(f"Invalid numerical value in holding data: {e}")

        # Extract enums with validation
        share_type = table.xpath('string(ns:shrsOrPrnAmt/ns:sshPrnamtType)', namespaces=ns).strip().upper()
        print("share type: ", share_type)
        if share_type not in ['SH', 'PRN']:
            raise ValueError(f"Invalid share type: {share_type}")

        investment_discretion = table.xpath('string(ns:investmentDiscretion)', namespaces=ns).strip().upper()
        print(investment_discretion)
        if investment_discretion not in ['SOLE', 'DFND', 'OTR']:
            raise ValueError(f"Invalid investment discretion: {investment_discretion}")

        put_call = table.xpath('string(ns:putCall)', namespaces=ns).strip().upper() or "NONE"
        print(put_call)
        if put_call not in ['NONE', 'PUT', 'CALL']:
            raise ValueError(f"Invalid put/call type: {put_call}")

        # Extract voting authority
        # voting = table.xpath('ns:votingAuthority', namespaces=ns)
        # print("voting: ", voting)
        sole = int(table.xpath('string(ns:votingAuthority/ns:Sole)', namespaces=ns).strip()) or 0
        shared = int(table.xpath('string(ns:votingAuthority/ns:Shared)', namespaces=ns).strip()) or 0
        none = int(table.xpath('string(ns:votingAuthority/ns:None)', namespaces=ns).strip()) or 0
        
        print(sole, shared, none)

        # security = get_or_create_security_class(conn, title_of_class)
        # holding_type = get_or_create_holding_type(conn, share_type)
        # option = get_or_create_option_type(conn, put_call)
        # discretion = get_or_create_discretion_type(conn, investment_discretion)
        # raise Exception("Fix me!")
    
    return []

def fetch_new_filing(_, cik_number, most_recent_date: dt.date):
    c = Company(cik_number)
    today_date = dt.date.today().strftime("%Y-%m-%d")

    for attempt in range(MAX_RETRIES):
        try:
            # Fetch filings from lastest filing date to today
            filings = c.get_filings(
                sort_by='filing_date',
                form=VALID_FORMS,
                filing_date=(f"{most_recent_date.strftime('%Y-%m-%d')}:{today_date}"),
            )

            if not filings:
                print(
                    f"No new valid filings found for CIK {cik_number} since {most_recent_date}"
                )
                return

            for filing in filings:
                print(filing.__dict__)
                # technically there is no need for this check because
                # we are fetching only valid forms
                if filing.form in VALID_FORMS:
                    filing_date = filing.filing_date

                    assert isinstance(filing_date, dt.date)

                    if filing_date > most_recent_date:
                        accession_number = filing.accession_number

                        # Check if the filing already exists in the database
                        # if filing_exists(conn, accession_number):
                        #     continue


                        # this should exist in database because
                        # we are fetching CIKs that we know
                        # manager = get_manager_by_cik(conn, cik_number)
                        # if not manager:
                        #     logger.error("Manager not found for CIK: %s", cik_number)
                        #     continue

                        # manager_id = manager.manager_id
                        # new_filing_id = insert_filing(conn, manager_id, filing)
                        # holding_inserted_count = 0

                        # insert holdings
                        # Looping through attachments to find XML files
                        for attachment in filing.attachments:
                            if (
                                attachment.document.endswith('.xml')
                                and 'primary_doc' not in attachment.document
                            ):
                                print(attachment.document)
                                assert attachment.content

                                # soup = BeautifulSoup(attachment.content, 'xml')
                                if isinstance(attachment.content, str):
                                    # Convert string to bytes
                                    xml_content = attachment.content.encode('utf-8')
                                else:
                                    xml_content = attachment.content

                                tree = etree.fromstring(xml_content)
                                nsmap = tree.nsmap
                                print(nsmap)

                                def find_namespaces(tree):
                                    all_elements = tree.xpath('//*')
                                    unique_prefixes = set()
                                    for elem in all_elements:
                                        if elem.prefix:
                                            unique_prefixes.add(elem.prefix)
                                    
                                    return list(unique_prefixes)

                                ns = {
                                    'ns': (
                                        nsmap[None]
                                        if None in nsmap
                                        else (nsmap['ns1'] if "ns1" in nsmap else nsmap[find_namespaces(tree)[0]])
                                    )
                                }
                                print("chosen: ", ns)
                                info_tables = tree.xpath(
                                    '//ns:infoTable', namespaces=ns
                                )
                                # info_tables = tree.xpath('//ns:informationTable', namespaces=ns)
                                print(info_tables)

                                for holding in gather_holdings_from_soup(None, info_tables, ns):
                                    pass

                                    # assert (
                                    #     holding._issuer_cusip
                                    # ), "CUSIP is required for each holding"
                                    # assert (
                                    #     holding._issuer_name
                                    # ), "Issuer name is required for each holding"

                                    # issuer = get_or_create_issuer(
                                    #     conn,
                                    #     holding._issuer_cusip,
                                    #     holding._issuer_name,
                                    # )

                                #     holding.issuer_id = issuer.issuer_id
                                #     holding.filing_id = new_filing_id
                                #     holdings.append(holding)

                                #     # insert_holding(conn, holding)
                                #     holding_inserted_count += 1

                                # if holdings:
                                #     insert_holdings_batch(conn, holdings)

                    else:
                        print(
                            f"Filing date {filing_date} is not newer than most recent date {most_recent_date} for CIK {cik_number}"
                        )
        except HTTPStatusError as e:
            # Handle HTTP errors, retry on server errors
            if (
                e.response.status_code in [500, 503, 504]
                and (attempt + 1) < MAX_RETRIES
            ):
                print(
                    f"HTTP error {e.response.status_code} for CIK {cik_number}, retrying..."
                )
                time.sleep(2 * (attempt + 1))
                continue
            print(e)

fetch_new_filing(None, 93751, dt.date(2021, 1, 1))

In [ ]:
from edgar import *

set_identity("test-autoingest@company.com")

c = Company("0002012383")

filings = c.get_filings(form=["13F-HR", "13F-HR/A", "13F-HR/A/A"])

In [ ]:
filings